# Test Technique Data Engineer : Health Data Hub

## 1. Chargement des données

In [1]:
import json
import pandas as pd
from pathlib import Path

SOURCE_PATH   = r"C:\Users\bench\Desktop\PSD\PROJET_COHORTE_2026\SITE_A"
DELIVERY_PATH = r"C:\Users\bench\Desktop\PSD\PROJET_COHORTE_2026_delivery\SITE_A"

def load_zone(base_path):
    """Charge tous les fichiers JSON d'une zone en extrayant le chemin ET le contenu.
    """
    base = Path(base_path)
    records = []
    for json_path in base.rglob("*.json"):
        rel_parts = json_path.relative_to(base).parts
        with open(json_path, encoding="utf-8") as f:
            data = json.load(f)
        records.append({
            "dir_patient": rel_parts[0],
            "dir_visit":   rel_parts[1],
            "file_name":   json_path.stem,
            **data
        })
    return pd.DataFrame(records)

df_source   = load_zone(SOURCE_PATH)
df_delivery = load_zone(DELIVERY_PATH)

print(f"Source   : {len(df_source)} fichiers chargés")
print(f"Delivery : {len(df_delivery)} fichiers chargés")

Source   : 6800 fichiers chargés
Delivery : 6800 fichiers chargés


## 2. Profiling

In [4]:
def profile_zone(df, name):
    print("\n\n")
    print(f" {name}")
    print(f"  Fichiers total          : {len(df)}")
    print(f"  Patients uniques        : {df['Patient_ID'].nunique()}")
    print(f"  Visites uniques         : {df.groupby(['Patient_ID','VISIT_ID']).ngroups}")
    print(f"  FILE_IDs uniques        : {df['FILE_ID'].nunique()}")
    print(f"  Doublons FILE_ID        : {df['FILE_ID'].duplicated().sum()}")
    print(f"  Valeurs nulles Patient_ID: {df['Patient_ID'].isnull().sum()}")
    print(f"  Catégories              : {sorted(df['Category'].unique())}")

profile_zone(df_source,   "SOURCE   (PROJET_COHORTE_2026)")
profile_zone(df_delivery, "DELIVERY (PROJET_COHORTE_2026_delivery)")




 SOURCE   (PROJET_COHORTE_2026)
  Fichiers total          : 6800
  Patients uniques        : 101
  Visites uniques         : 608
  FILE_IDs uniques        : 6800
  Doublons FILE_ID        : 0
  Valeurs nulles Patient_ID: 0
  Catégories              : ['TYPE_A', 'TYPE_B', 'TYPE_C']



 DELIVERY (PROJET_COHORTE_2026_delivery)
  Fichiers total          : 6800
  Patients uniques        : 81
  Visites uniques         : 588
  FILE_IDs uniques        : 6800
  Doublons FILE_ID        : 0
  Valeurs nulles Patient_ID: 0
  Catégories              : ['TYPE_A', 'TYPE_B', 'TYPE_C']


In [5]:
# Comparaison directe source vs delivery
summary = pd.DataFrame({
    "Métrique": ["Fichiers", "Patients", "Visites", "FILE_IDs uniques"],
    "Source": [
        len(df_source),
        df_source['Patient_ID'].nunique(),
        df_source.groupby(['Patient_ID','VISIT_ID']).ngroups,
        df_source['FILE_ID'].nunique()
    ],
    "Delivery": [
        len(df_delivery),
        df_delivery['Patient_ID'].nunique(),
        df_delivery.groupby(['Patient_ID','VISIT_ID']).ngroups,
        df_delivery['FILE_ID'].nunique()
    ]
})
summary["Écart"] = summary["Delivery"] - summary["Source"]
summary

,Métrique,Source,Delivery,Écart
0,Fichiers,6800,6800,0
1,Patients,101,81,-20
2,Visites,608,588,-20
3,FILE_IDs uniques,6800,6800,0


## 3. Investigation — Hypothèse 1 : fichiers manquants ?

**Hypothèse :** La delivery est incomplète, certains fichiers de la source sont absents

In [7]:
source_file_ids   = set(df_source['FILE_ID'])
delivery_file_ids = set(df_delivery['FILE_ID'])

in_source_not_delivery = source_file_ids - delivery_file_ids
in_delivery_not_source = delivery_file_ids - source_file_ids

print(f"FILE_IDs présents en source mais ABSENTS en delivery : {len(in_source_not_delivery)}")
print(f"FILE_IDs présents en delivery mais ABSENTS en source : {len(in_delivery_not_source)}")
print()


FILE_IDs présents en source mais ABSENTS en delivery : 0
FILE_IDs présents en delivery mais ABSENTS en source : 0



**Aucun fichier manquant, tous les FILE_IDs sont présents dans les deux zones et donc l'incomplétude signalée n'est PAS une perte de fichiers, c'est un problème de classification**

## 4. Investigation — Hypothèse 2 : patients disparus = mauvaise attribution ?

**Hypothèse :** Les fichiers existent bien, mais le `Patient_ID` a été écrasé lors de la transformation.

In [8]:
source_pids   = set(df_source['Patient_ID'])
delivery_pids = set(df_delivery['Patient_ID'])

missing_patients = source_pids - delivery_pids
new_patients     = delivery_pids - source_pids

print(f"Patients présents en source mais ABSENTS en delivery : {len(missing_patients)}")
print(f"Patients présents en delivery mais ABSENTS en source : {len(new_patients)}")
print()

# Combien de fichiers source appartiennent aux patients manquants ?
missing_files_df = df_source[df_source['Patient_ID'].isin(missing_patients)]
print(f"Fichiers source appartenant aux {len(missing_patients)} patients manquants : {len(missing_files_df)}")

# Ces fichiers sont-ils dans la delivery malgré tout ?
found_in_delivery = missing_files_df[missing_files_df['FILE_ID'].isin(delivery_file_ids)]
print(f"Parmi eux, présents dans la delivery (sous un autre Patient_ID ?) : {len(found_in_delivery)}")

Patients présents en source mais ABSENTS en delivery : 20
Patients présents en delivery mais ABSENTS en source : 0

Fichiers source appartenant aux 20 patients manquants : 1228
Parmi eux, présents dans la delivery (sous un autre Patient_ID ?) : 1228


In [9]:
# Identifier vers quel Patient_ID delivery ces fichiers ont été redirigés
delivery_by_fid = df_delivery.set_index('FILE_ID')[['Patient_ID','VISIT_ID','dir_patient']]

# Joindre pour comparer source vs delivery
comparison = missing_files_df[['FILE_ID','Patient_ID','VISIT_ID']].copy()
comparison.columns = ['FILE_ID','Patient_ID_source','VISIT_ID_source']
comparison = comparison.join(delivery_by_fid, on='FILE_ID')
comparison.rename(columns={'Patient_ID':'Patient_ID_delivery','VISIT_ID':'VISIT_ID_delivery'}, inplace=True)

print("Patients delivery vers lesquels les fichiers ont été redirigés :")
print(comparison['Patient_ID_delivery'].value_counts())

Patients delivery vers lesquels les fichiers ont été redirigés :
Patient_ID_delivery
28342286687094591113440599999    1228
Name: count, dtype: int64


**ANOMALIE CONFIRMÉE : 1228 fichiers appartenant à 20 patients différents ont été tous rattachés au MÊME patient dans la delivery !**

## 5. Analyse de la cause racine — Corruption dans la zone SOURCE

**Hypothèse :** Le bug vient de fichiers "intrus" dans la source avec un mauvais `Patient_ID`.

In [16]:
# Dans la source : combien de dossiers patient (hash) contiennent plusieurs Patient_IDs ?
pid_per_dir = df_source.groupby('dir_patient')['Patient_ID'].nunique()
contaminated_dirs = pid_per_dir[pid_per_dir > 1]

print(f"Dossiers source contenant plusieurs Patient_IDs distincts : {len(contaminated_dirs)}")
print()

# Détail : combien de fichiers intrus par dossier ?
SINK_PID = comparison['Patient_ID_delivery'].value_counts().index[0]
print(f"Patient_ID 'gouffre' (SINK) : {SINK_PID}")
print()

Dossiers source contenant plusieurs Patient_IDs distincts : 20

Patient_ID 'gouffre' (SINK) : 28342286687094591113440599999



In [17]:
# Tableau récapitulatif des dossiers contaminés
contamination = []
for dir_hash in contaminated_dirs.index:
    dir_files = df_source[df_source['dir_patient'] == dir_hash]
    pid_counts = dir_files['Patient_ID'].value_counts()
    legit_pid = pid_counts[pid_counts.index != SINK_PID].index[0] if len(pid_counts) > 1 else pid_counts.index[0]
    contamination.append({
        "dir_hash": dir_hash,
        "patient_légitime (tronqué)": legit_pid[:20] + "...",
        "nb_fichiers_légitimes": pid_counts.get(legit_pid, 0),
        "nb_fichiers_intrus": pid_counts.get(SINK_PID, 0)
    })

df_contamination = pd.DataFrame(contamination)
print(df_contamination.to_string(index=False))

    dir_hash patient_légitime (tronqué)  nb_fichiers_légitimes  nb_fichiers_intrus
0ac75c23d24d    69057672297956333362...                     43                   1
2e4d1e6a1f13    11502237552726980289...                     41                   1
3e955f14ff22    90047300010981578923...                     18                   1
601f4ba3d3b9    95605857113713551330...                    103                   1
60fb1d00d41f    76560732319014282187...                     82                   1
6317c0b9b13f    24857820227257994494...                     61                   1
6b7c349fb24e    26002810618252247788...                     48                   1
70026ce83abe    20749856108543483466...                    101                   1
79ac24b41099    36998333663606768534...                     33                   1
8adcad38d670    88439392593211826657...                     22                   1
9205767bce65    90345529551425707183...                     28                   1
9d64

## 6. Vérification de la règle métier : cohérence dossier ↔ contenu JSON

**Règle :** `Patient_ID` et `VISIT_ID` dans le JSON doivent correspondre au chemin du dossier.

In [18]:
# Dans la delivery : dir_patient == Patient_ID et dir_visit == VISIT_ID ?
delivery_pid_ok  = (df_delivery['dir_patient'] == df_delivery['Patient_ID']).sum()
delivery_vid_ok  = (df_delivery['dir_visit']   == df_delivery['VISIT_ID']).sum()

print(f"Delivery — fichiers où dir_patient == Patient_ID (JSON) : {delivery_pid_ok}/{len(df_delivery)} ")
print(f"Delivery — fichiers où dir_visit   == VISIT_ID   (JSON) : {delivery_vid_ok}/{len(df_delivery)} ")


Delivery — fichiers où dir_patient == Patient_ID (JSON) : 6800/6800 
Delivery — fichiers où dir_visit   == VISIT_ID   (JSON) : 6800/6800 


**Dans la delivery, la structure des dossiers est cohérente avec les JSON.** 

**Le problème : les JSON eux-mêmes ont un Patient_ID incorrect (écrasé par le script)**

## 7. Résumé & Conclusion

### Ce qui a été trouvé

| Observation | Valeur |
|---|---|
| Fichiers total (source = delivery) | 6 800 |
| Patients source | 101 |
| Patients delivery | 81 |
| Patients manquants | **20** |
| Fichiers mal attribués | **1 228** |
| Patient "gouffre" dans delivery | 1 seul |
| Fichiers intrus dans la source | **20** (1 par dossier affecté) |
| FILE_IDs perdus | 0 |

### Cause racine

La zone source contient **20 fichiers "intrus"** : des fichiers placés dans un mauvais dossier patient, portant le `Patient_ID` du patient SINK (`28342286687094591113...`). Un fichier intrus est présent dans chacun des 20 dossiers affectés.

Le script de transformation a probablement lu le premier fichier rencontré dans chaque dossier pour en déduire le `Patient_ID` du dossier entier. Quand ce premier fichier était un intrus, l'ensemble du dossier (toutes les visites, tous les fichiers) a été attribué au patient SINK.

### Impact métier

- **20 patients fantômes** : absents de la delivery, introuvables pour les chercheurs.
- **109 visites inaccessibles** sous le bon identifiant patient.
- **1 228 fichiers mal classés** sous un unique patient, créant un dossier patient anormalement volumineux.
- Toute analyse ou requête filtrée par `Patient_ID` retournera des résultats faux pour ces patients.

### Recommandations

1. **Court terme** : corriger la delivery en reclassant les 1 228 fichiers dans le bon dossier patient (reconstruction possible à 100% via `FILE_ID` ↔ source).
2. **Correction du script** : ne pas déduire le `Patient_ID` d'un dossier à partir d'un seul fichier — vérifier que tous les fichiers d'un dossier partagent le même `Patient_ID`, et lever une alerte sinon.
3. **Contrôle qualité source** : identifier et corriger les 20 fichiers intrus dans la source (mauvais dossier d'origine).